In [4]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:22px;}
</style>
"""))

# 1. 데이터셋가져오기

In [5]:
import pandas as pd
url='https://raw.githubusercontent.com/4aix/data/refs/heads/master/ch13_apt_fillna_median.csv'
df=pd.read_csv(url)
df.head()

,지역명,평당분양가격,연도,월
0,서울,18189.0,2013,12
1,부산,8111.0,2013,12
2,대구,8080.0,2013,12
3,인천,10204.0,2013,12
4,광주,6098.0,2013,12


In [7]:
# #아래 라벨인코딩방식으로 하면 cost가 줄어들지 않는다. 지역명이 너무 작아서.
# - 지역명2:지역병필드를 라벨인코딩하여 추가(df)
# - 독립변수(X):지역명2,연도,월
# - 종속변수(y):평당분양가격

# #독립변수와 종속변수의 스케일조정. 즉 종속변수를 reshape해서 2차원으로 바꿔줌
# #라이브러리사용, 독립변수 3개를 2차원 & 종속변수를 reshape
#     * 정규화작업 후:지역명2m,연도m,월m 컬럼으로 추가(df) 그리고 평당분양가격m =>df_m(따로만들수도 있음)
#     * 표준화작업 후:지역명2s,연도s,월s 컬럼으로 추가(df) 그리고 평당분양가격m =>df_s(따로만들수도 있음) 즉,
# =>지역명,연도,월,지역명2,지역명2m,연도m,월m,평당분양가격m,지역명2s,연도s,월s컬럼,평당분양가격s
# - 데이터프레임.to_numpy(), 데이터프레임.values, np.array(데이터프레임) 등을 이용하여 데이터프레임을 넘파이 배열로 변환

# 2. 지역명의 라벨인코딩

In [8]:
from sklearn.preprocessing import LabelEncoder

# 라벨 인코더 생성
encoder = LabelEncoder()

# '지역명'을 숫자로 변환하여 새 열에 저장
df['지역명_라벨'] = encoder.fit_transform(df['지역명'])

# 지역명과 라벨 번호의 대응 관계 확인
label_mapping = dict(zip(encoder.classes_,
                         encoder.transform(encoder.classes_)))

print(label_mapping)
df[['지역명', '지역명_라벨']].head()

{'강원': 0, '경기': 1, '경남': 2, '경북': 3, '광주': 4, '대구': 5, '대전': 6, '부산': 7, '서울': 8, '세종': 9, '울산': 10, '인천': 11, '전남': 12, '전북': 13, '제주': 14, '충남': 15, '충북': 16}


,지역명,지역명_라벨
0,서울,8
1,부산,7
2,대구,5
3,인천,11
4,광주,4


In [3]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# 실제 지역명 입력
region_names = [
    '서울', '부산', '대구', '인천', '광주', '대전', '울산',
    '경기', '세종', '강원', '충북', '충남', '전북', '전남',
    '경북', '경남', '제주'
]

df_region = pd.DataFrame({'지역명': region_names})

# 라벨 인코딩
encoder = LabelEncoder()
df_region['지역명_라벨'] = encoder.fit_transform(df_region['지역명'])

print(df_region)

   지역명  지역명_라벨
0   서울       8
1   부산       7
2   대구       5
3   인천      11
4   광주       4
5   대전       6
6   울산      10
7   경기       1
8   세종       9
9   강원       0
10  충북      16
11  충남      15
12  전북      13
13  전남      12
14  경북       3
15  경남       2
16  제주      14


# 2-1 numpy배열이 되으로 원본보전

In [10]:
#to_numpy() 등으로 변환하면 결과는 별도의 데이터프레임이 아니라 NumPy 배열이됨. 그래서 원본을 보존하려면 먼저 복사본을 만들기.

from sklearn.preprocessing import LabelEncoder

# 원본 데이터프레임 보존
encoded_df = df.copy()

# 복사본의 지역명만 라벨 인코딩
encoder = LabelEncoder()
encoded_df['지역명'] = encoder.fit_transform(encoded_df['지역명'])

# 데이터프레임을 NumPy 배열로 변환
data = encoded_df.to_numpy()

print(encoded_df.head())
print(data[:5])
print(type(data))
# <class 'numpy.ndarray'>

   지역명   평당분양가격    연도   월  지역명_라벨
0    8  18189.0  2013  12       8
1    7   8111.0  2013  12       7
2    5   8080.0  2013  12       5
3   11  10204.0  2013  12      11
4    4   6098.0  2013  12       4
[[8.0000e+00 1.8189e+04 2.0130e+03 1.2000e+01 8.0000e+00]
 [7.0000e+00 8.1110e+03 2.0130e+03 1.2000e+01 7.0000e+00]
 [5.0000e+00 8.0800e+03 2.0130e+03 1.2000e+01 5.0000e+00]
 [1.1000e+01 1.0204e+04 2.0130e+03 1.2000e+01 1.1000e+01]
 [4.0000e+00 6.0980e+03 2.0130e+03 1.2000e+01 4.0000e+00]]
<class 'numpy.ndarray'>


In [11]:
from sklearn.preprocessing import LabelEncoder

# 라벨 인코더 생성
encoder = LabelEncoder()

# 기존 '지역명'은 보존하고 '지역명2' 열 추가
df['지역명2'] = encoder.fit_transform(df['지역명'])

# 독립변수(X): 지역명2, 연도, 월
X = df[['지역명2', '연도', '월']].to_numpy()

# 종속변수(y): 평당분양가격
y = df['평당분양가격'].to_numpy()

# 결과 확인
print(df[['지역명', '지역명2', '연도', '월', '평당분양가격']].head())

print('\n독립변수 X:')
print(X[:5])

print('\n종속변수 y:')
print(y[:5])

print('\n자료형:')
print(type(X))
print(type(y))

print('\n배열 크기:')
print('X 크기:', X.shape)
print('y 크기:', y.shape)

   지역명  지역명2    연도   월   평당분양가격
0    8     8  2013  12  18189.0
1    7     7  2013  12   8111.0
2    5     5  2013  12   8080.0
3   11    11  2013  12  10204.0
4    4     4  2013  12   6098.0

독립변수 X:
[[   8 2013   12]
 [   7 2013   12]
 [   5 2013   12]
 [  11 2013   12]
 [   4 2013   12]]

종속변수 y:
[18189.  8111.  8080. 10204.  6098.]

자료형:
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>

배열 크기:
X 크기: (2176, 3)
y 크기: (2176,)


# 3. MinMaxScaling

In [12]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# X: 지역명2, 연도, 월
X = df[['지역명2', '연도', '월']].to_numpy()

# y: 평당분양가격
y = df['평당분양가격'].to_numpy()

# X와 y에 사용할 스케일러를 각각 생성
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

# 독립변수 스케일링
X_scaled = scaler_X.fit_transform(X)

# y는 1차원 배열이므로 2차원으로 변환한 후 스케일링
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

# 결과 확인용 데이터프레임
scaled_df = pd.DataFrame(
    X_scaled,
    columns=['지역명2_scaled', '연도_scaled', '월_scaled']
)

scaled_df['평당분양가격_scaled'] = y_scaled

print(scaled_df.head())
print('\nX_scaled 배열:')
print(X_scaled[:5])

print('\ny_scaled 배열:')
print(y_scaled[:5])

   지역명2_scaled  연도_scaled  월_scaled  평당분양가격_scaled
0       0.5000        0.0       1.0       0.328198
1       0.4375        0.0       1.0       0.065274
2       0.3125        0.0       1.0       0.064466
3       0.6875        0.0       1.0       0.119878
4       0.2500        0.0       1.0       0.012757

X_scaled 배열:
[[0.5    0.     1.    ]
 [0.4375 0.     1.    ]
 [0.3125 0.     1.    ]
 [0.6875 0.     1.    ]
 [0.25   0.     1.    ]]

y_scaled 배열:
[[0.32819817]
 [0.06527439]
 [0.06446563]
 [0.11987843]
 [0.01275746]]


# 4. StandardScaling

In [13]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 독립변수(X): 지역명2, 연도, 월
X = df[['지역명2', '연도', '월']].to_numpy()

# 종속변수(y): 평당분양가격
y = df['평당분양가격'].to_numpy()

# X와 y에 사용할 스케일러를 각각 생성
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# 독립변수 표준화
X_scaled = scaler_X.fit_transform(X)

# y를 2차원으로 변환한 후 표준화
y_scaled = scaler_y.fit_transform(
    y.reshape(-1, 1)
)

# 결과 확인용 데이터프레임
standard_df = pd.DataFrame(
    X_scaled,
    columns=['지역명2_scaled', '연도_scaled', '월_scaled']
)

standard_df['평당분양가격_scaled'] = y_scaled.reshape(-1)

print(standard_df.head())

   지역명2_scaled  연도_scaled  월_scaled  평당분양가격_scaled
0     0.000000  -1.875367   1.62196       1.168591
1    -0.204124  -1.875367   1.62196      -0.728312
2    -0.612372  -1.875367   1.62196      -0.734147
3     0.612372  -1.875367   1.62196      -0.334363
4    -0.816497  -1.875367   1.62196      -1.107203


# 4-1 새롭게 만들어진 DataFrame(원본 및 정규화,표준화 포함)

In [ ]:
from sklearn.preprocessing import (
    LabelEncoder,
    MinMaxScaler,
    StandardScaler
)

# --------------------------------------------------
# 1. 지역명 라벨 인코딩
# --------------------------------------------------
encoder = LabelEncoder()

# 원래 지역명은 보존하고 지역명2에 인코딩 결과 저장
df['지역명2'] = encoder.fit_transform(df['지역명'])

# 원본 독립변수와 종속변수
X = df[['지역명2', '연도', '월']].to_numpy()
y = df['평당분양가격'].to_numpy().reshape(-1, 1)


# --------------------------------------------------
# 2. Min-Max 정규화: 접미사 m
# --------------------------------------------------
minmax_X = MinMaxScaler()
minmax_y = MinMaxScaler()

X_m = minmax_X.fit_transform(X)
y_m = minmax_y.fit_transform(y)

df[['지역명2m', '연도m', '월m']] = X_m
df['평당분양가격m'] = y_m.reshape(-1)


# --------------------------------------------------
# 3. Standard 표준화: 접미사 s
# --------------------------------------------------
standard_X = StandardScaler()
standard_y = StandardScaler()

X_s = standard_X.fit_transform(X)
y_s = standard_y.fit_transform(y)

df[['지역명2s', '연도s', '월s']] = X_s
df['평당분양가격s'] = y_s.reshape(-1)


# --------------------------------------------------
# 4. 요청한 열로 별도의 데이터프레임 생성
# --------------------------------------------------
result_columns = [
    '지역명',
    '연도',
    '월',
    '지역명2',
    '지역명2m',
    '연도m',
    '월m',
    '평당분양가격m',
    '지역명2s',
    '연도s',
    '월s',
    '평당분양가격s'
]

result_df = df[result_columns].copy()

print(result_df.head())


# --------------------------------------------------
# 5. 데이터프레임을 NumPy 배열로 변환
# --------------------------------------------------
data = result_df.to_numpy()

print('\nNumPy 배열:')
print(data[:5])

print('\n자료형:', type(data))
print('배열 크기:', data.shape)
print('배열 dtype:', data.dtype)

# 5. 지역명을 원핫인코딩

In [15]:
import pandas as pd

# 지역명 원핫 인코딩
region_onehot = pd.get_dummies(
    df['지역명'],
    prefix='지역명',
    dtype=int
)

# 원본 df에 원핫 인코딩 열 추가
df_onehot = pd.concat(
    [df, region_onehot],
    axis=1
)

print(region_onehot.head())
print(df_onehot.head())

   지역명_0  지역명_1  지역명_2  지역명_3  지역명_4  지역명_5  지역명_6  지역명_7  지역명_8  지역명_9  \
0      0      0      0      0      0      0      0      0      1      0   
1      0      0      0      0      0      0      0      1      0      0   
2      0      0      0      0      0      1      0      0      0      0   
3      0      0      0      0      0      0      0      0      0      0   
4      0      0      0      0      1      0      0      0      0      0   

   지역명_10  지역명_11  지역명_12  지역명_13  지역명_14  지역명_15  지역명_16  
0       0       0       0       0       0       0       0  
1       0       0       0       0       0       0       0  
2       0       0       0       0       0       0       0  
3       0       1       0       0       0       0       0  
4       0       0       0       0       0       0       0  
   지역명   평당분양가격    연도   월  지역명_라벨  지역명2   지역명2m  연도m   월m   평당분양가격m  ...  \
0    8  18189.0  2013  12       8     8  0.5000  0.0  1.0  0.328198  ...   
1    7   8111.0  2013  12       7    